In [ ]:
# In[1]:

import json

# Load the task.json file to understand the alert context
task_file_path = 'rca100/cases/t070/task.json'

with open(task_file_path, 'r') as file:
    task_data = json.load(file)

alert_title = task_data.get('alert_title', None)
alert_window = task_data.get('alert_window', None)
alert_entity = task_data.get('alert_entity', None)
prompt_text = task_data.get('prompt_text', None)

alert_title, alert_window, alert_entity, prompt_text

```
Out[1]:
```
The alert is titled "frontend响应时间突增告警" (frontend response time spike alert). It occurred between 2026-03-17 01:54:52 and 2026-03-17 02:00:09 (UTC+8). The entity that triggered the alert is the operation "frontend::POST /api/checkout" in the APM domain. The user is requesting an analysis of the root cause.

The original code execution output of IPython Kernel is also provided below for reference:

('frontend响应时间突增告警', {'start': '2026-03-17T01:54:52.021789+08:00', 'end': '2026-03-17T02:00:09+08:00'}, {'entity_id': 'd5479ffd165e089a67b3d4eeb8519fb7', 'entity_name': 'frontend::POST /api/checkout', 'entity_type': 'apm.operation', 'entity_domain': 'apm'}, '<code vibeops_object type="alert_event">frontend响应时间突增告警<alert_event event_id="c5f0542b0a4d76b59fb8f06d0c3cfbee" trans_id="udf5j2j43h05368b2u3p72dth2" rule_id="bd9b5c12-f227-49d9-b3bd-9c6515ee906d" rule_name="frontend响应时间突增告警" /></code> 帮我分析下根因。')```
```

In [ ]:
# In[2]:

import pandas as pd

# Load the metrics.parquet file
metrics_file_path = 'rca100/cases/t070/metrics.parquet'
metrics_df = pd.read_parquet(metrics_file_path)

# List all available KPIs and their associated components
kpi_components = metrics_df[['metric', 'entity_name']].drop_duplicates()

kpi_components

```
Out[2]:
```
The `metrics.parquet` file contains 250 unique KPI-component pairs. These include KPIs like `node_ready_status`, `node_memory_usage_rate`, `node_disk_usage_rate`, and `error_rate`, associated with various components such as nodes, services, and operations (e.g., `cart::POST /oteldemo.CartService/EmptyCart`, `frontend-proxy::ingress`). To analyze anomalies, focus on KPIs relevant to the alert entity, such as `frontend::POST /api/checkout`.

The original code execution output of IPython Kernel is also provided below for reference:

metric                                 entity_name
0           node_ready_status                                            
1485    node_not_ready_status                                            
1890   node_memory_usage_rate                                            
5940   node_pod_running_count                                            
7560     node_disk_usage_rate                                            
...                       ...                                         ...
79874              error_rate  cart::POST /oteldemo.CartService/EmptyCart
79888              error_rate                     frontend-proxy::ingress
79894              error_rate             frontend-proxy-67d64dc4f8-4ptg6
79896              error_rate             frontend-proxy-67d64dc4f8-8c99b
79898              error_rate             frontend-proxy-67d64dc4f8-xz2fd

[250 rows x 2 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pytz
from datetime import datetime

# Convert time column to numeric for calculations
metrics_df['time'] = pd.to_numeric(metrics_df['time'], errors='coerce')
metrics_df['value'] = pd.to_numeric(metrics_df['value'], errors='coerce')

# Calculate global thresholds (P95) for each KPI-component pair
thresholds = metrics_df.groupby(['metric', 'entity_name'])['value'].quantile(0.95).reset_index()
thresholds.rename(columns={'value': 'global_p95'}, inplace=True)

# Filter data within the alert time window
alert_start = datetime.fromisoformat('2026-03-17T01:54:52.021789+08:00').astimezone(pytz.timezone('Asia/Shanghai'))
alert_end = datetime.fromisoformat('2026-03-17T02:00:09+08:00').astimezone(pytz.timezone('Asia/Shanghai'))
alert_start_us = int(alert_start.timestamp() * 1_000_000)  # Convert to microseconds
alert_end_us = int(alert_end.timestamp() * 1_000_000)      # Convert to microseconds

filtered_metrics = metrics_df[
    (metrics_df['time'] >= alert_start_us) & (metrics_df['time'] <= alert_end_us)
]

# Focus on KPIs relevant to the alert entity and other possible root cause components
alert_entity_name = 'frontend::POST /api/checkout'
relevant_components = ['frontend', 'frontend-proxy', 'checkout', 'payment', 'cart', 'inventory', 'product-catalog']
filtered_relevant_metrics = filtered_metrics[
    filtered_metrics['entity_name'].str.contains('|'.join(relevant_components), case=False, na=False)
]

thresholds, filtered_relevant_metrics

```
Out[3]:
```
Global thresholds (P95) were calculated for 250 KPI-component pairs across the system. The filtered metrics data within the alert time window includes 1,305 rows, focusing on KPIs relevant to the alert entity `frontend::POST /api/checkout` and other potential root cause components like `frontend-proxy`, `cart`, and `payment`. These metrics will help identify anomalies contributing to the alert.

The original code execution output of IPython Kernel is also provided below for reference:

(                                  metric      entity_name   global_p95
0       apiserver_concurrent_connections                      6.300000
1     apiserver_non_2xx_read_request_qps                      0.088318
2    apiserver_non_2xx_write_request_qps                      0.026471
3          apiserver_requests_per_second                     26.728133
4              arms_grouped_thread_count        inventory     1.291262
..                                   ...              ...          ...
245                             workload          payment   405.300000
246                             workload  product-catalog  2866.400000
247                             workload            quote   406.000000
248                             workload   recommendation   249.900000
249                             workload         shipping   811.300000

[250 rows x 3 columns],                    time domain     entity_set                         entity_id                                 entity_name       metric       value                            metric_set_id service
55080  1773683722000000    apm  apm.operation  3ff65eef282fcdd303aebab1c132fe9f  cart::POST /oteldemo.CartService/EmptyCart  error_count   57.000000  apm@metric_set@apm.metric.apm.operation    None
55081  1773683752000000    apm  apm.operation  3ff65eef282fcdd303aebab1c132fe9f  cart::POST /oteldemo.CartService/EmptyCart  error_count  247.000000  apm@metric_set@apm.metric.apm.operation    None
55082  1773683782000000    apm  apm.operation  3ff65eef282fcdd303aebab1c132fe9f  cart::POST /oteldemo.CartService/EmptyCart  error_count  325.000000  apm@metric_set@apm.metric.apm.operation    None
55083  1773683812000000    apm  apm.operation  3ff65eef282fcdd303aebab1c132fe9f  cart::POST /oteldemo.CartService/EmptyCart  error_count  260.000000  apm@metric_set@apm.metric.apm.operation    None
55084  1773683842000000    apm  apm.operation  3ff65eef282fcdd303aebab1c132fe9f  cart::POST /oteldemo.CartService/EmptyCart  error_count  351.000000  apm@metric_set@apm.metric.apm.operation    None
...                 ...    ...            ...                               ...                                         ...          ...         ...                                      ...     ...
79895  1773683842000000    apm   apm.instance  eab531e9b188fe065edbebdbf839c609             frontend-proxy-67d64dc4f8-4ptg6   error_rate    0.012903   apm@metric_set@apm.metric.apm.instance    None
79896  1773683812000000    apm   apm.instance  fa548039d9a7b76db9ec70507e734faf             frontend-proxy-67d64dc4f8-8c99b   error_rate    0.008130   apm@metric_set@apm.metric.apm.instance    None
79897  1773683842000000    apm   apm.instance  fa548039d9a7b76db9ec70507e734faf             frontend-proxy-67d64dc4f8-8c99b   error_rate    0.008072   apm@metric_set@apm.metric.apm.instance    None
79902  1773683812000000    apm   apm.instance  d1f1b0349fadcfdc23ca3918ac448623             frontend-proxy-67d64dc4f8-xz2fd   error_rate    0.008222   apm@metric_set@apm.metric.apm.instance    None
79903  1773683842000000    apm   apm.instance  d1f1b0349fadcfdc23ca3918ac448623             frontend-proxy-67d64dc4f8-xz2fd   error_rate    0.007874   apm@metric_set@apm.metric.apm.instance    None

[1305 rows x 9 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

# Merge the filtered metrics data with the global thresholds to identify anomalies
merged_data = filtered_relevant_metrics.merge(
    thresholds, 
    on=['metric', 'entity_name'], 
    how='left'
)

# Identify anomalies where the value exceeds the global P95 threshold
anomalies = merged_data[merged_data['value'] > merged_data['global_p95']]

# Report anomalies for each component and KPI
anomalies_grouped = anomalies.groupby(['entity_name', 'metric']).size().reset_index(name='anomaly_count')

anomalies, anomalies_grouped

```
Out[4]:
```
Anomaly detection identified 273 data points exceeding global thresholds across various KPIs and components. Key anomalies include:

- `frontend::GET` with high `request_count`.
- `cart::POST /oteldemo.CartService/GetCart` with high `request_count`.
- `frontend-proxy` instances with elevated `error_rate`.

Additionally, anomalies were found in KPIs like `cpu`, `latency`, and `mem` for components such as `cart` and `product-catalog`. These anomalies suggest potential performance issues in these components during the alert time window.

The original code execution output of IPython Kernel is also provided below for reference:

(                  time domain          entity_set                         entity_id                               entity_name         metric        value                            metric_set_id service   global_p95
32    1773683782000000    apm       apm.operation  28887014538b3aeed572d8e4810f6d88                             frontend::GET  request_count  2242.000000  apm@metric_set@apm.metric.apm.operation    None  2159.400000
35    1773683872000000    apm       apm.operation  28887014538b3aeed572d8e4810f6d88                             frontend::GET  request_count  2336.000000  apm@metric_set@apm.metric.apm.operation    None  2159.400000
36    1773683902000000    apm       apm.operation  28887014538b3aeed572d8e4810f6d88                             frontend::GET  request_count  2425.000000  apm@metric_set@apm.metric.apm.operation    None  2159.400000
51    1773683752000000    apm       apm.operation  7bb1b1512922edfdefac66f31c640a81  cart::POST /oteldemo.CartService/GetCart  request_count  1150.000000  apm@metric_set@apm.metric.apm.operation    None  1134.000000
52    1773683782000000    apm       apm.operation  7bb1b1512922edfdefac66f31c640a81  cart::POST /oteldemo.CartService/GetCart  request_count  1178.000000  apm@metric_set@apm.metric.apm.operation    None  1134.000000
...                ...    ...                 ...                               ...                                       ...            ...          ...                                      ...     ...          ...
1284  1773683992000000    apm  apm.service.legacy                                                             product-catalog        latency     0.000063                                     None    None     0.000061
1297  1773683812000000    apm       apm.operation  d1c2b3981457bf9b5c5beefbbf246ebc                   frontend-proxy::ingress     error_rate     0.009554  apm@metric_set@apm.metric.apm.operation    None     0.009488
1299  1773683812000000    apm        apm.instance  eab531e9b188fe065edbebdbf839c609           frontend-proxy-67d64dc4f8-4ptg6     error_rate     0.013405   apm@metric_set@apm.metric.apm.instance    None     0.013380
1301  1773683812000000    apm        apm.instance  fa548039d9a7b76db9ec70507e734faf           frontend-proxy-67d64dc4f8-8c99b     error_rate     0.008130   apm@metric_set@apm.metric.apm.instance    None     0.008127
1303  1773683812000000    apm        apm.instance  d1f1b0349fadcfdc23ca3918ac448623           frontend-proxy-67d64dc4f8-xz2fd     error_rate     0.008222   apm@metric_set@apm.metric.apm.instance    None     0.008135

[273 rows x 10 columns],                                           entity_name                       metric  anomaly_count
0                                                cart                          cpu              3
1                                                cart                        error              3
2                                                cart                      latency              5
3                                                cart                          mem              7
4                                                cart                         slow              5
..                                                ...                          ...            ...
88                    product-catalog-79f7b8c4b-lmp9q  avg_request_latency_seconds              3
89                    product-catalog-79f7b8c4b-lmp9q                request_count              1
90  product-catalog::oteldemo.ProductCatalogServic...  avg_request_latency_seconds              4
91  product-catalog::oteldemo.ProductCatalogServic...                request_count              2
92  product-catalog::oteldemo.ProductCatalogServic...                request_count              3

[93 rows x 3 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

# Load the traces.parquet file
traces_file_path = 'rca100/cases/t070/traces.parquet'
traces_df = pd.read_parquet(traces_file_path)

# Convert time columns to numeric for filtering
traces_df['startTime'] = pd.to_numeric(traces_df['startTime'], errors='coerce')
traces_df['endTime'] = pd.to_numeric(traces_df['endTime'], errors='coerce')

# Filter traces within the alert time window
filtered_traces = traces_df[
    (traces_df['startTime'] >= alert_start_us * 1_000) &  # Convert microseconds to nanoseconds
    (traces_df['endTime'] <= alert_end_us * 1_000)
]

# Focus on traces involving the `frontend::POST /api/checkout` operation
frontend_checkout_traces = filtered_traces[
    filtered_traces['spanName'].str.contains('frontend::POST /api/checkout', case=False, na=False)
]

# Identify the most downstream faulty service in the call chain
downstream_faulty_traces = frontend_checkout_traces[
    (frontend_checkout_traces['statusCode'] != 0) | 
    (frontend_checkout_traces['statusMessage'].str.contains('error', case=False, na=False))
]

downstream_faulty_services = downstream_faulty_traces.groupby('serviceName').size().reset_index(name='fault_count')

frontend_checkout_traces, downstream_faulty_services

```
Out[5]:
```
No traces involving the `frontend::POST /api/checkout` operation were found within the alert time window (2026-03-17 01:54:52 to 2026-03-17 02:00:09 UTC+8). Consequently, no downstream faulty services could be identified from the trace data.

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [traceId, spanId, parentSpanId, kind, spanName, startTime, endTime, duration, serviceName, pid, hostname, statusCode, statusMessage, traceState, resources, attributes, events, __topic__, __tag__:__pack_id__, ip, links]
Index: []

[0 rows x 21 columns], Empty DataFrame
Columns: [serviceName, fault_count]
Index: [])```
```

In [ ]:
# In[6]:

# Load the logs.parquet file
logs_file_path = 'rca100/cases/t070/logs.parquet'
logs_df = pd.read_parquet(logs_file_path)

# Convert the `_time_` column to numeric for filtering
logs_df['_time_'] = pd.to_numeric(logs_df['_time_'], errors='coerce')

# Filter logs within the alert time window
filtered_logs = logs_df[
    (logs_df['_time_'] >= alert_start_us) & 
    (logs_df['_time_'] <= alert_end_us)
]

# Search for error-related messages or patterns in relevant components
relevant_components = ['frontend', 'cart', 'frontend-proxy', 'product-catalog']
error_logs = filtered_logs[
    (filtered_logs['content'].str.contains('error', case=False, na=False)) &
    (filtered_logs['_container_name_'].str.contains('|'.join(relevant_components), case=False, na=False))
]

error_logs

```
Out[6]:
```
No error-related log messages were found within the alert time window (2026-03-17 01:54:52 to 2026-03-17 02:00:09 UTC+8) for the `frontend`, `cart`, `frontend-proxy`, or `product-catalog` components.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [content, _time_, _source_, _namespace_, _pod_uid_, _container_ip_, _image_name_, _container_name_, _pod_name_, __topic__, __tag__:__pack_id__, __tag__:__hostname__, __tag__:_node_name_, __tag__:_node_ip_, __tag__:_cluster_id_]
Index: []

[0 rows x 15 columns]```
```